# 00 — Tạo feature, ghép lịch và tạo target theo horizon

Toàn bộ phép xử lý nằm trong notebook. Đọc snapshot modeling 2005–2026 và warm-up 2004;
không dùng EDA 2010–2026 làm đầu vào. Không tự crawl. Run All tái tạo features, bảng ghép
và dữ liệu H1/H5/H10/H20; không train model. Sau khi đổi dữ liệu phải chạy lại notebook
classification và ARM để kết quả phản ánh đúng dữ liệu mới.

Output đã lưu được kiểm tra bằng một lần tái tạo ở thư mục kiểm chứng riêng. Code mặc định
ghi vào `data/modeling`; biến môi trường PROJECT_FINAL_DATA_OUTPUT chỉ phục vụ kiểm chứng
ở output riêng, không thay đổi nguồn raw/support.

In [1]:
from pathlib import Path
import os
import json
import hashlib
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'project_manifest.json').exists())
DATASET_DIR = Path(os.environ.get('PROJECT_FINAL_DATA_OUTPUT', ROOT/'data/modeling'))
RAW_DIR = ROOT/'data/modeling/raw'
OUTPUT_DIR = DATASET_DIR/'features'
TARGET = 'Gold_Target_Next'
SOURCE_FEATURES = {
 'gold':['Daily_Return','Return_Lag_1','Intraday_Range','Volatility_20','Distance_to_MA_20','RSI_14','MACD_Histogram'],
 'oil':['Price_Change'], 'dollar_index':['Daily_Return'], 'sp500':['Daily_Return','Drawdown']}
SOURCES = list(SOURCE_FEATURES)
PREFIX = dict(zip(SOURCES,['Gold','Oil','Dollar','SP500']))
FEATURES = [f'{PREFIX[s]}_{name}' for s in SOURCES for name in SOURCE_FEATURES[s]]
print('Nguồn raw:',RAW_DIR)
print('Output:',DATASET_DIR)

Nguồn raw: D:\1_TaiLieuHocTap\2627\DataMining\PROJECT_FINAL\data\modeling\raw
Output: C:\Users\Nguyen Tri Nhan\.codex\visualizations\2026\09\25\01a0d85a-eec5-79a2-878c-eaac75741d84\final_rebuild


## 1. Tiện ích lưu và kiểm tra
Các hàm ngắn nằm ngay tại đây; không import xử lý từ src.

In [2]:
def require(condition, message):
    print(f'{"PASS" if condition else "FAIL"}: {message}')
    if not condition:
        raise ValueError(message)

def finite(frame):
    return not frame.isna().any().any() and not frame.select_dtypes('number').isin([float('inf'), -float('inf')]).any().any()

def save_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, date_format='%Y-%m-%d')

def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

## 2. Feature trên lịch riêng của từng nguồn

Giữ nguyên công thức của bản đã chốt: 7 feature Gold, Oil signed price change, Dollar return,
S&P return và drawdown. Warm-up 2004 chỉ hỗ trợ tính chỉ báo, không thành mẫu nghiên cứu.
Các nguồn ngoài Gold được tính feature trước khi carry-forward theo lịch Gold.

In [3]:
def load_original(source):
    frame = pd.read_csv(RAW_DIR / f'{source}.csv', skiprows=[1, 2]).rename(columns={'Price': 'Date'})
    frame['Date'] = pd.to_datetime(frame['Date'])
    frame = frame.sort_values('Date').reset_index(drop=True)
    require(not frame.empty and not frame.Date.duplicated().any() and finite(frame), f'{source}: original raw valid')
    return frame

def load_raw(source):
    original = load_original(source)
    history = pd.read_csv(ROOT / f'data/modeling/support/{source}_warmup.csv', parse_dates=['Date'])
    require(history.Date.max() < original.Date.min(), f'{source}: warm-up strictly before original data')
    frame = pd.concat([history[original.columns], original], ignore_index=True)
    require(not frame.Date.duplicated().any() and frame.Date.is_monotonic_increasing and finite(frame), f'{source}: extended raw valid')
    return frame

def add_features(source, raw):
    frame = raw.copy()
    if source == 'oil':
        # Không chia cho giá Oil: giữ đúng dấu khi Close âm.
        frame['Price_Change'] = frame['Close'].diff()
    else:
        if (frame['Close'] <= 0).any():
            raise ValueError(f'{source}: Close không dương cần được xác minh.')
        frame['Daily_Return'] = frame['Close'].pct_change(fill_method=None)
        if source == 'sp500':
            frame['Drawdown'] = frame['Close'] / frame['Close'].cummax() - 1
        if source == 'gold':
            frame['Return_Lag_1'] = frame['Daily_Return'].shift(1)
            frame['Intraday_Range'] = (frame['High'] - frame['Low']) / frame['Close']
            frame['Volatility_20'] = frame['Daily_Return'].rolling(20, min_periods=20).std(ddof=1)
            # MA20 là biến trung gian, không xuất như một feature riêng.
            ma20 = frame['Close'].rolling(20, min_periods=20).mean()
            frame['Distance_to_MA_20'] = frame['Close'] / ma20 - 1
            change = frame['Close'].diff()
            gains, losses = change.clip(lower=0), -change.clip(upper=0)
            avg_gain = pd.Series(float('nan'), index=frame.index)
            avg_loss = pd.Series(float('nan'), index=frame.index)
            # Seed RSI bằng trung bình 14 thay đổi đầu, sau đó Wilder recursion.
            if len(frame) > 14:
                avg_gain.loc[14] = gains.loc[1:14].mean()
                avg_loss.loc[14] = losses.loc[1:14].mean()
                for position in range(15, len(frame)):
                    avg_gain.loc[position] = (13 * avg_gain.loc[position - 1] + gains.loc[position]) / 14
                    avg_loss.loc[position] = (13 * avg_loss.loc[position - 1] + losses.loc[position]) / 14
            strength = avg_gain / avg_loss.where(avg_loss != 0)
            frame['RSI_14'] = 100 - 100 / (1 + strength)
            frame.loc[(avg_loss == 0) & (avg_gain > 0), 'RSI_14'] = 100
            frame.loc[(avg_loss == 0) & (avg_gain == 0), 'RSI_14'] = 50
            # MACD/Signal chỉ dùng nội bộ để tính Histogram.
            fast = frame['Close'].ewm(span=12, adjust=False, min_periods=12).mean()
            slow = frame['Close'].ewm(span=26, adjust=False, min_periods=26).mean()
            macd = fast - slow
            signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
            frame['MACD_Histogram'] = macd - signal
    features = SOURCE_FEATURES[source]
    return frame[list(raw.columns) + features], features

In [4]:
def build_features():
    for source in SOURCES:
        raw = load_raw(source)
        frame, features = add_features(source, raw)
        require(frame[raw.columns].equals(raw), f'{source}: giữ nguyên giá trị raw')
        save_csv(frame, OUTPUT_DIR / f'{source}.csv')
        print(f'{source}: {len(frame):,} dòng, {len(features)} feature, giữ NaN warm-up.')

In [5]:
build_features()
print('11 feature đã tạo trên lịch từng nguồn.')

PASS: gold: original raw valid
PASS: gold: warm-up strictly before original data
PASS: gold: extended raw valid


PASS: gold: giữ nguyên giá trị raw
gold: 5,707 dòng, 7 feature, giữ NaN warm-up.
PASS: oil: original raw valid
PASS: oil: warm-up strictly before original data
PASS: oil: extended raw valid
PASS: oil: giữ nguyên giá trị raw


oil: 5,710 dòng, 1 feature, giữ NaN warm-up.
PASS: dollar_index: original raw valid
PASS: dollar_index: warm-up strictly before original data
PASS: dollar_index: extended raw valid
PASS: dollar_index: giữ nguyên giá trị raw
dollar_index: 5,727 dòng, 1 feature, giữ NaN warm-up.
PASS: sp500: original raw valid
PASS: sp500: warm-up strictly before original data
PASS: sp500: extended raw valid
PASS: sp500: giữ nguyên giá trị raw


sp500: 5,714 dòng, 2 feature, giữ NaN warm-up.
11 feature đã tạo trên lịch từng nguồn.


## 3. Ghép dữ liệu và ghi nhật ký lịch

Giữ lịch hợp ở bảng ghép; nguồn ngoài Gold dùng quan sát gần nhất trong quá khứ.
Không điền Gold và không tự tạo sample có nhãn khi raw không có Gold. Lưu các ngày thiếu
Gold và từng giá trị carry-forward để truy vết. Modeling giữ đủ 5.458 phiên Gold.
Target H1 bằng 1 nếu phiên sau tăng, bằng 0 khi giảm hoặc đứng giá.

In [6]:
def build_merged():
    output = DATASET_DIR / 'merged'
    audit = output / 'alignment_logs'
    originals = {source: load_original(source) for source in SOURCES}
    tables = {source: pd.read_csv(DATASET_DIR / f'features/{source}.csv', parse_dates=['Date']) for source in SOURCES}
    start = min(frame.Date.min() for frame in originals.values())
    end = max(frame.Date.max() for frame in originals.values())
    union = sorted(set().union(*(set(frame.Date) for frame in originals.values())))
    merged = pd.DataFrame({'Date': union})
    fills = []
    # Feature values were computed on each source's real sessions, before alignment.
    for source in SOURCES:
        prefix = PREFIX[source]
        frame = tables[source].copy()
        frame = frame.loc[frame.Date.le(end)]
        original_dates = set(originals[source].Date)
        observed = merged.Date.isin(original_dates)
        if source == 'gold':
            aligned = frame.set_index('Date').reindex(merged.Date).reset_index()
            source_dates = aligned.Date.where(observed)
        else:
            frame['Source_Date'] = frame.Date
            aligned = pd.merge_asof(merged[['Date']], frame.sort_values('Date'), on='Date', direction='backward')
            source_dates = aligned.pop('Source_Date')
            for index in merged.index[~observed & source_dates.notna()]:
                for column in originals[source].columns.drop('Date').tolist() + SOURCE_FEATURES[source]:
                    fills.append({'Date': merged.loc[index, 'Date'], 'Source': source,
                                  'Source_Date': source_dates.loc[index],
                                  'Age_calendar_days': (merged.loc[index, 'Date'] - source_dates.loc[index]).days,
                                  'Column': f'{prefix}_{column}', 'Filled_value': aligned.loc[index, column],
                                  'Reason': 'No source observation on union date; last available source session'})
        for column in aligned.columns.drop('Date'):
            merged[f'{prefix}_{column}'] = aligned[column].values
        merged[f'{prefix}_Observed'] = observed.values
        merged[f'{prefix}_Source_Date'] = source_dates.values
        merged[f'{prefix}_Age_Days'] = (merged.Date - source_dates).dt.days.values
        merged[f'{prefix}_Carried_Forward'] = ((~observed) & source_dates.notna()).values
    gold = originals['gold']
    support = pd.read_csv(ROOT / 'data/modeling/support/gold_horizon_support.csv', parse_dates=['Date'])
    support = support.loc[support.Date.gt(gold.Date.max())].sort_values('Date').head(1)
    require(len(support) == 1, 'One actual next Gold session available for the final label')
    extended = pd.concat([gold[['Date', 'Close']], support[['Date', 'Close']]], ignore_index=True)
    future_close = extended.Close.shift(-1)
    target = (future_close > extended.Close).astype('Int64').where(future_close.notna())
    target_audit = pd.DataFrame({'Date':extended.Date, 'Close':extended.Close,
                                 'Next_Gold_Date':extended.Date.shift(-1), 'Next_Gold_Close':future_close,
                                 TARGET:target, 'Flat_next_session':future_close.eq(extended.Close)})
    target_audit = target_audit.loc[target_audit.Date.isin(gold.Date)].copy()
    merged = merged.merge(target_audit[['Date', TARGET]], on='Date', how='left', validate='one_to_one')
    merged[TARGET] = merged[TARGET].astype('Int64')
    merged['Model_Eligible'] = merged.Gold_Observed & merged[TARGET].notna() & merged[FEATURES].notna().all(axis=1)
    merged['Exclusion_Reason'] = ''
    merged.loc[~merged.Gold_Observed, 'Exclusion_Reason'] = 'No Gold observation in raw snapshot on this date'
    merged.loc[merged.Gold_Observed & merged[TARGET].isna(), 'Exclusion_Reason'] = 'Next Gold close unavailable'
    merged.loc[merged.Gold_Observed & merged[TARGET].notna() & merged[FEATURES].isna().any(axis=1), 'Exclusion_Reason'] = 'Required feature unavailable'
    model = merged.loc[merged.Model_Eligible, ['Date'] + FEATURES + [TARGET]].copy()
    require(merged.Date.tolist() == union, 'Outer calendar retains every original date')
    require(len(model) == len(gold), 'Every original Gold session has usable features and target')
    require(finite(model) and set(model[TARGET]) <= {0,1}, 'Model data finite, target 0/1')
    save_csv(merged, output / 'date_cleaned_dataset.csv')
    save_csv(model, output / 'modeling_dataset.csv')
    save_csv(pd.DataFrame({'Candidate_feature': FEATURES}), output / 'candidate_features.csv')
    save_csv(pd.DataFrame(fills), audit / 'forward_fill_log.csv')
    save_csv(target_audit, audit / 'gold_target_trace.csv')
    save_csv(merged[['Date','Model_Eligible','Exclusion_Reason']], audit / 'model_eligibility.csv')
    # Keep a separate inventory of dates absent from the original Gold snapshot.
    # Absence is not proof that the exchange was closed or the crawl was complete.
    missing_gold = merged.loc[~merged.Gold_Observed,
        ['Date', 'Gold_Observed', 'Oil_Observed', 'Dollar_Observed', 'SP500_Observed',
         'Model_Eligible', 'Exclusion_Reason']].copy()
    missing_gold['Retained_in_outer_dataset'] = True
    missing_gold['Resolution'] = 'Keep in outer dataset; exclude from labeled modeling; do not fill Gold or infer target'
    missing_gold['Cause_status'] = 'Unverified: calendar difference or source/crawl omission'
    save_csv(missing_gold, audit / 'dates_without_gold.csv')
    save_csv(merged[['Date']+[c for c in merged if c.endswith(('_Observed','_Source_Date','_Age_Days','_Carried_Forward'))]], audit / 'calendar_alignment.csv')
    save_json({'study_start':str(start.date()), 'study_end':str(end.date()), 'outer_rows':len(merged),
               'gold_original_rows':len(gold), 'model_rows':len(model),
               'non_gold_dates':int((~merged.Gold_Observed).sum()),
               'flat_sessions_retained':int(target_audit.Flat_next_session.sum()),
               'target_counts':{str(k):int(v) for k,v in model[TARGET].value_counts().items()},
               'final_gold_label_support_date':str(support.Date.iloc[0].date()),
               'filled_source_dates':len(pd.DataFrame(fills)[['Date','Source']].drop_duplicates()) if fills else 0,
               'original_raw_sha256':{s:hashlib.sha256((ROOT/f'data/modeling/raw/{s}.csv').read_bytes()).hexdigest() for s in SOURCES},
               'policy':'Outer union; external backward as-of; no Gold fill; no flat removal; no model loss on Gold calendar'}, audit / 'summary.json')
    print(f'Outer calendar: {len(merged)} rows; modeling: {len(model)} rows; no original Gold sessions lost.')

In [7]:
build_merged()
print('Bảng modeling vừa tạo:')
display(pd.read_csv(DATASET_DIR/'merged/modeling_dataset.csv').head())

PASS: gold: original raw valid
PASS: oil: original raw valid
PASS: dollar_index: original raw valid
PASS: sp500: original raw valid


PASS: One actual next Gold session available for the final label
PASS: Outer calendar retains every original date
PASS: Every original Gold session has usable features and target
PASS: Model data finite, target 0/1


Outer calendar: 5473 rows; modeling: 5458 rows; no original Gold sessions lost.
Bảng modeling vừa tạo:


,Date,Gold_Daily_Return,Gold_Return_Lag_1,Gold_Intraday_Range,Gold_Volatility_20,Gold_Distance_to_MA_20,Gold_RSI_14,Gold_MACD_Histogram,Oil_Price_Change,Dollar_Daily_Return,SP500_Daily_Return,SP500_Drawdown,Gold_Target_Next
0,2005-01-03,-0.020114,0.003440,0.000000,0.011550,-0.028034,38.127619,-1.448093,-1.330002,0.005566,-0.008119,-0.009452,0
1,2005-01-04,-0.000467,-0.020114,0.000000,0.011026,-0.025450,37.963545,-1.772142,1.790001,0.015621,-0.011671,-0.021013,0
2,2005-01-05,-0.004434,-0.000467,0.000000,0.011029,-0.026716,36.362742,-2.004979,-0.520000,-0.000363,-0.003628,-0.024564,0
3,2005-01-06,-0.013127,-0.004434,0.000000,0.011252,-0.036084,32.070381,-2.399996,2.170002,0.007390,0.003506,-0.021145,0
4,2005-01-07,-0.004988,-0.013127,0.011936,0.008925,-0.038879,30.611112,-2.647127,-0.130001,0.005532,-0.001431,-0.022546,1


## 4. Target H1/H5/H10/H20 trên lịch phiên Gold gốc

Gold support 21/09–16/10/2026 chỉ cung cấp giá tương lai cho nhãn cuối; không thêm mẫu đầu vào.
Target tạo trên lịch Gold gốc trước khi chọn ngày modeling. Không đổi feature giữa horizon.
Purge Train/Val diễn ra ở notebook model/ARM khi chia tập, không xóa dòng khỏi dữ liệu nguồn.

In [8]:
gold = load_original('gold')
support = pd.read_csv(ROOT/'data/modeling/support/gold_horizon_support.csv', parse_dates=['Date'])
support = support.loc[support.Date.gt(gold.Date.max()), ['Date','Close']]
extended = pd.concat([gold[['Date','Close']], support], ignore_index=True).sort_values('Date').reset_index(drop=True)
require(not extended.Date.duplicated().any(), 'Lịch Gold mở rộng không trùng ngày')
modeling = pd.read_csv(DATASET_DIR/'merged/modeling_dataset.csv', parse_dates=['Date'])
for h in [1,5,10,20]:
    trace = extended[['Date','Close']].copy()
    trace['Target_Date'] = extended.Date.shift(-h)
    trace['Target_Close'] = extended.Close.shift(-h)
    trace = trace.loc[trace.Date.isin(gold.Date)].copy()
    require(trace[['Target_Date','Target_Close']].notna().all().all(), f'H{h}: đủ giá để tạo target')
    trace[TARGET] = (trace.Target_Close > trace.Close).astype(int)
    trace['Horizon'] = h
    frame = modeling.drop(columns=TARGET).merge(trace[['Date',TARGET]], on='Date', validate='one_to_one')
    require(frame.Date.tolist() == modeling.Date.tolist(), f'H{h}: giữ nguyên lịch modeling')
    save_csv(trace, DATASET_DIR/f'horizons/h{h}/target_trace.csv')
    save_csv(frame, DATASET_DIR/f'horizons/h{h}/modeling_dataset.csv')
    print(f'H{h}: {len(frame):,} mẫu; target cuối {trace.Target_Date.max().date()}')
print('Tiếp theo: Classification H1/H5/H10/H20, hoặc ARM-H1/H5/H10/H20 rồi ARM-FINAL.')

PASS: gold: original raw valid
PASS: Lịch Gold mở rộng không trùng ngày
PASS: H1: đủ giá để tạo target
PASS: H1: giữ nguyên lịch modeling


H1: 5,458 mẫu; target cuối 2026-09-21
PASS: H5: đủ giá để tạo target
PASS: H5: giữ nguyên lịch modeling
H5: 5,458 mẫu; target cuối 2026-09-25
PASS: H10: đủ giá để tạo target
PASS: H10: giữ nguyên lịch modeling


H10: 5,458 mẫu; target cuối 2026-10-02
PASS: H20: đủ giá để tạo target
PASS: H20: giữ nguyên lịch modeling
H20: 5,458 mẫu; target cuối 2026-10-16
Tiếp theo: Classification H1/H5/H10/H20, hoặc ARM-H1/H5/H10/H20 rồi ARM-FINAL.
